# Experiment 4: Handling imbalanced data

Five techniques, applied to the training split only: class weights, SMOTE oversampling, ADASYN,
random undersampling and SMOTE-ENN. TF-IDF trigrams, 1,000 features, same Random Forest.

In [1]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import seaborn as sns
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000'))
EXPERIMENT = 'Exp 4 - Handling Imbalanced Data'
mlflow.set_experiment(EXPERIMENT)
BATCH = datetime.now().strftime('%Y%m%d-%H%M%S')

NGRAM_RANGE = (1, 3)
MAX_FEATURES = 1000

df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)

# Vectorizer is fitted on the training split only
vectorizer = TfidfVectorizer(ngram_range=NGRAM_RANGE, max_features=MAX_FEATURES)
X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec = vectorizer.transform(X_test_text)
X_train_vec.shape

2026/09/11 11:05:12 INFO mlflow.tracking.fluent: Experiment with name 'Exp 4 - Handling Imbalanced Data' does not exist. Creating a new experiment.


(29329, 1000)

In [2]:
def log_evaluation(y_true, y_pred, title):
    """Log accuracy, per-class metrics and a confusion matrix to the active MLflow run."""
    mlflow.log_metric('accuracy', accuracy_score(y_true, y_pred))
    for label, metrics in classification_report(y_true, y_pred, output_dict=True).items():
        if isinstance(metrics, dict):
            mlflow.log_metrics({f'{label}_{name}': value for name, value in metrics.items()})

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set(xlabel='Predicted', ylabel='Actual', title=f'Confusion Matrix: {title}')
    mlflow.log_figure(fig, 'confusion_matrix.png')
    plt.close(fig)


def compare_runs():
    """Table of this batch's runs; the video reads the same numbers off MLflow's parallel-coordinates plot."""
    runs = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string=f"tags.batch = '{BATCH}'")
    columns = {
        'tags.mlflow.runName': 'run',
        'metrics.accuracy': 'accuracy',
        'metrics.-1_precision': 'neg_precision',
        'metrics.-1_recall': 'neg_recall',
        'metrics.1_precision': 'pos_precision',
        'metrics.1_recall': 'pos_recall',
    }
    return runs[list(columns)].rename(columns=columns).sort_values('accuracy', ascending=False).round(4)

In [3]:
RESAMPLERS = {
    'oversampling': SMOTE(random_state=42),
    'adasyn': ADASYN(random_state=42),
    'undersampling': RandomUnderSampler(random_state=42),
    'smote_enn': SMOTEENN(random_state=42),
}


def run_imbalanced_experiment(imbalance_method):
    X_train, y_train_bal = X_train_vec, y_train
    class_weight = 'balanced' if imbalance_method == 'class_weights' else None
    if imbalance_method in RESAMPLERS:
        X_train, y_train_bal = RESAMPLERS[imbalance_method].fit_resample(X_train_vec, y_train)

    with mlflow.start_run(run_name=f'Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams'):
        mlflow.set_tags({'experiment_type': 'imbalance_handling', 'model_type': 'RandomForestClassifier', 'batch': BATCH})
        mlflow.log_params({
            'vectorizer_type': 'TF-IDF',
            'ngram_range': NGRAM_RANGE,
            'vectorizer_max_features': MAX_FEATURES,
            'n_estimators': 200,
            'max_depth': 15,
            'imbalance_method': imbalance_method,
            'train_rows_after_resampling': X_train.shape[0],
        })

        model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1, class_weight=class_weight)
        model.fit(X_train, y_train_bal)
        log_evaluation(y_test, model.predict(X_test_vec), f'imbalance={imbalance_method}')


for method in ['class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn']:
    run_imbalanced_experiment(method)

2026/09/11 11:05:20 INFO mlflow.tracking._tracking_service.client: 🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigrams at: http://127.0.0.1:5000/#/experiments/4/runs/3f6859922985436ea332de101ad8e90e.


2026/09/11 11:05:20 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4.


2026/09/11 11:05:27 INFO mlflow.tracking._tracking_service.client: 🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigrams at: http://127.0.0.1:5000/#/experiments/4/runs/7d42c1a2f2af45b2a783f658de52c07e.


2026/09/11 11:05:27 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4.


2026/09/11 11:05:44 INFO mlflow.tracking._tracking_service.client: 🏃 View run Imbalance_adasyn_RandomForest_TFIDF_Trigrams at: http://127.0.0.1:5000/#/experiments/4/runs/b6f5b3f3ebb94cc1bd4c2850d6a5cabb.


2026/09/11 11:05:44 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4.


2026/09/11 11:05:45 INFO mlflow.tracking._tracking_service.client: 🏃 View run Imbalance_undersampling_RandomForest_TFIDF_Trigrams at: http://127.0.0.1:5000/#/experiments/4/runs/1e6d4db1477945c2a3cf1e63f091bbf7.


2026/09/11 11:05:45 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4.


2026/09/11 11:06:25 INFO mlflow.tracking._tracking_service.client: 🏃 View run Imbalance_smote_enn_RandomForest_TFIDF_Trigrams at: http://127.0.0.1:5000/#/experiments/4/runs/1d32ec5e0aa74625a059bf7e1a23f198.


2026/09/11 11:06:25 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4.


In [4]:
compare_runs()

,run,accuracy,neg_precision,neg_recall,pos_precision,pos_recall
2,Imbalance_adasyn_RandomForest_TFIDF_Trigrams,0.6772,0.6228,0.4164,0.8357,0.6049
4,Imbalance_class_weights_RandomForest_TFIDF_Tri...,0.6753,0.6084,0.4455,0.8351,0.5958
1,Imbalance_undersampling_RandomForest_TFIDF_Tri...,0.6730,0.6015,0.4739,0.8507,0.5726
3,Imbalance_oversampling_RandomForest_TFIDF_Trig...,0.6713,0.5877,0.4648,0.8394,0.5850
0,Imbalance_smote_enn_RandomForest_TFIDF_Trigrams,0.4296,0.3042,0.8552,1.0000,0.0257


The video picks **SMOTE oversampling**: it balances precision and recall across all three classes best.
Experiments 5 and 6 train every model on SMOTE-oversampled TF-IDF trigrams with 1,000 features.